In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

In [2]:
from Data_preparation import create_fish_pipeline, prepare_fish_data


In [3]:
def evaluate_random_forest(X_train, y_train, X_dev, y_dev):
    print("Evaluating Random Forest Regressor...")

    param_grid = {
        'algo__n_estimators': [2000],
        'algo__max_depth': [2, 3, 4],
        'algo__min_samples_split': [2, 5]
    }

    pipeline = create_fish_pipeline()

    pipeline_with_algo = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('algo', RandomForestRegressor(random_state=42))
    ])

    grid_search = GridSearchCV(
        pipeline_with_algo, param_grid,
        cv=5,  # 3-fold cross-validation
        scoring='r2',  # Use R² as the evaluation metric
        verbose=1  # Show progress in terminal
    )
    grid_search.fit(X_train, y_train)

    # This shows us our best model based on cross-validation R² score.
    best_estimator = grid_search.best_estimator_

    # 📊 FEATURE IMPORTANCE SECTION
    try:
        model = best_estimator.named_steps["algo"]
        preprocessor = best_estimator.named_steps["preprocessor"]
        feature_names = preprocessor.get_feature_names_out()
        importances = model.feature_importances_

        feature_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("\nTop 10 Most Important Features:")
        print(feature_df.head(10))
    except Exception as e:
        print("Could not extract feature importances:", e)

    # We are making predicitons on the dev set here
    y_pred = best_estimator.predict(X_dev)

    # Here we are calculating the following values
    mse = mean_squared_error(y_dev, y_pred)
    mae = mean_absolute_percentage_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    # Shows you the best performance from the training phase and the hyperparameters that gave it.
    print("Grid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, mse, mae, r2

In [4]:
# Step 1: Prepare fish data (split into train/dev/test)
X_train, X_dev, X_test, y_train, y_dev, y_test = prepare_fish_data(ratios=((1/10), (1/10)))

# Step 2: Run hyperparameter tuning on train/dev sets
best_model, dev_rmse, dev_mape, dev_r2 = evaluate_random_forest(X_train, y_train, X_dev, y_dev)

   # 🔍 DEBUG: Show the sizes of the splits
print("✅ Data Split Shapes:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)
print("  X_test:", X_test.shape)
print("  y_train:", y_train.shape)
print("  y_dev:", y_dev.shape)
print("  y_test:", y_test.shape)

# Step 1: Make predictions
y_train_pred = best_model.predict(X_train)
y_dev_pred = best_model.predict(X_dev)

# Step 2: Define a reusable function to evaluate metrics
def evaluate_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_target = np.mean(y_true)
    print(f"\n📊 {label} Set Performance:")
    print(f"Mean of y_{label.lower()}: {mean_target:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}")
    print(f"R²: {r2:.4f}")
    return rmse, mape, r2

# Step 3: Print evaluation metrics
train_rmse, train_mape, train_r2 = evaluate_metrics(y_train, y_train_pred, "Train")
dev_rmse, dev_mape, dev_r2 = evaluate_metrics(y_dev, y_dev_pred, "Dev")

Evaluating Random Forest Regressor...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Top 10 Most Important Features:
                         Feature  Importance
0           num__Spring Temp (F)    0.194193
10  num__Spring Temp (F) (Lag 3)    0.141933
8               num__Day of Year    0.089076
9                  num__Fish Age    0.081619
5                    num__# fish    0.079245
21          num__PM Transparency    0.060375
1              num__Max air temp    0.052875
7       num__Max Air Temp x Rain    0.040050
4               num__Calmar Rain    0.039741
2              num__Min air temp    0.037541
Grid searching is done!
Best score (neg MSE): 0.12420998805346847
Best hyperparameters:
{'algo__max_depth': 3, 'algo__min_samples_split': 2, 'algo__n_estimators': 2000}
✅ Data Split Shapes:
  X_train: (20756, 26)
  X_dev: (2594, 26)
  X_test: (2594, 26)
  y_train: (20756,)
  y_dev: (2594,)
  y_test: (2594,)

📊 Train Set Performance:
Mean of y_train: 99.9757
RMSE: 0.2274
M